## 1 创建python虚拟环境

In [1]:
%pip install -q "uv==0.13.6"
!uv --version
!uv python install 3.12.13
!if [ ! -d .venv ]; then uv venv --managed-python --python 3.12.13 .venv; else echo ".venv 已存在，跳过创建"; fi

uv 0.13.6 (x86_64-unknown-linux-gnu)
Installed Python 3.12.13 in 1.03s
 + cpython-3.12.13-linux-x86_64-gnu (python3.12)
Using CPython 3.12.13
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


In [2]:
%%script .venv/bin/python

import sys
import platform

print("Python 完整版本：", sys.version)
print("Python 路径：", sys.executable)
print("虚拟环境目录：", sys.prefix)
print("基础 Python 目录：", sys.base_prefix)
print("系统架构：", platform.machine())

Python 完整版本： 3.12.13 (main, Aug  7 2026, 02:25:39) [Clang 22.1.3 ]
Python 路径： /content/.venv/bin/python
虚拟环境目录： /content/.venv
基础 Python 目录： /root/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu
系统架构： x86_64


In [3]:
!nvidia-smi
!apt-get install -y ffmpeg
!ffmpeg -version | head -n 1
!uv pip install \
    --python /content/.venv/bin/python \
    --torch-backend cu128 \
    "torch==2.11.0+cu128" \
    "torchvision==0.26.0+cu128" \
    "torchcodec==0.11.0+cu128" \
    "numpy==2.0.2" \
    "nvidia-npp-cu12==12.4.1.87"

Fri Oct 30 09:23:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             57W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
# 在当前 Notebook 内核中配置 NPP 动态库路径，供后续子进程继承。
import os
from pathlib import Path

# 指向虚拟环境内 NPP 的库目录；该路径与 /content/.venv 和 Python 3.12 对应。
npp_dir = Path(
    "/content/.venv/lib/python3.12/site-packages/nvidia/npp/lib"
)

# 检查目标动态库是否存在
libs = list(npp_dir.glob("libnppicc.so*"))
print("NPP 库文件:", libs)

# 未找到目标库时立即报错，提示先检查依赖安装。
assert libs, "NPP 库未找到，请检查安装结果"

# 为后续 %%script 启动的 Python 子进程配置动态库搜索路径
# 保留已有路径；Linux 下多个动态库搜索目录使用冒号分隔。
old_path = os.environ.get("LD_LIBRARY_PATH", "")
paths = old_path.split(":") if old_path else []

# 将 NPP 目录放在搜索路径前面，并避免重复运行时重复添加。
if str(npp_dir) not in paths:
    os.environ["LD_LIBRARY_PATH"] = (
        str(npp_dir) + (":" + old_path if old_path else "")
    )

print("NPP 动态库路径配置成功")

NPP 库文件: [PosixPath('/content/.venv/lib/python3.12/site-packages/nvidia/npp/lib/libnppicc.so.12')]
NPP 动态库路径配置成功


In [5]:
%%script /content/.venv/bin/python

# 在虚拟环境子进程中检查依赖版本，并执行一次 GPU 矩阵乘法。
# 先运行单元格 4，使此子进程继承配置好的 NPP 动态库搜索路径。
import torch
import torchvision
import numpy as np
import torchcodec

# 打印实际导入的库版本，便于核对安装结果。
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("TorchCodec:", torchcodec.__version__)
print("NumPy:", np.__version__)

# 这里的 CUDA 版本是 PyTorch 构建所用的版本；is_available() 检查 CUDA 是否可用。
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

# 在 GPU 上创建 1024×1024 的随机矩阵并相乘；CUDA 不可用时这里会报错。
x = torch.randn(1024, 1024, device="cuda")
y = x @ x

# 输出结果形状，用于基本冒烟检查；此处没有显式同步 GPU 或校验数值精度。
print("GPU 计算测试通过:", y.shape)

PyTorch: 2.11.0+cu128
torchvision: 0.26.0+cu128
TorchCodec: 0.11.0+cu128
NumPy: 2.0.2
CUDA: 12.8
GPU available: True
GPU 计算测试通过: torch.Size([1024, 1024])


## 2.下载Lerobot源码安装

In [6]:
!git clone --branch v0.6.1 --depth 1 https://github.com/huggingface/lerobot.git /content/lerobot
!git -C /content/lerobot describe --tags --exact-match
!git -C /content/lerobot rev-parse HEAD

Cloning into '/content/lerobot'...
remote: Enumerating objects: 1187, done.
remote: Counting objects: 100% (1187/1187), done.
remote: Compressing objects: 100% (1041/1041), done.
remote: Total 1187 (delta 153), reused 811 (delta 107), pack-reused 0 (from 0)
Receiving objects: 100% (1187/1187), 6.35 MiB | 13.47 MiB/s, done.
Resolving deltas: 100% (153/153), done.
Note: switching to '7e241bd630a3719a56157a497ce5d08f244784f1'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

v0.6.1
7e241bd630a3719a56157a497ce5d08f244784f1


In [7]:
!uv pip install \
    --python /content/.venv/bin/python \
    --torch-backend cu128 \
    -e "/content/lerobot[smolvla,training]" \
    "torch==2.11.0+cu128" \
    "torchvision==0.26.0+cu128" \
    "torchcodec==0.11.0+cu128" \
    "numpy==2.0.2" \
    "nvidia-npp-cu12==12.4.1.87"

!uv pip check --python /content/.venv/bin/python

Resolved 108 packages in 1.13s                                       
Prepared 76 packages in 2.11s                                            
Uninstalled 2 packages in 3ms
Installed 76 packages in 91ms                               
 + accelerate==1.15.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.15.1
 + attrs==26.1.0
 + av==15.1.0
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + cmake==4.1.3
 + datasets==4.8.5
 + dill==0.4.1
 + docopt==0.6.2
 + draccus==0.11.6
 + einops==0.8.2
 + farama-notifications==0.0.6
 + frozenlist==1.8.0
 - fsspec==2026.9.0
 + fsspec==2026.2.0
 + gitdb==4.0.12
 + gitpython==3.2.1
 + gymnasium==1.4.0
 + h11==0.16.0
 + hf-xet==1.7.1
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.33.0
 + idna==3.20
 + jsonlines==4.0.0
 + lerobot==0.6.1 (from file:///content/lerobot)
 + markdown-it-py==4.2.0
 + mdurl==0.1.2
 + mergedeep==1.3.

## 3.下载libero数据集

In [8]:
%%script /content/.venv/bin/python

from huggingface_hub import HfApi, snapshot_download
from pathlib import Path
import json

repo_id = "lerobot/libero"
dataset_dir = Path("/content/datasets/libero")

revision = HfApi().repo_info(
    repo_id=repo_id,
    repo_type="dataset"
).sha

print("Dataset:", repo_id)
print("Revision:", revision)

local_path = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    revision=revision,
    local_dir=str(dataset_dir),
    max_workers=8,
)

(dataset_dir / "dataset_revision.txt").write_text(revision)

print("\n下载目录:", local_path)

with open(dataset_dir / "meta/info.json") as f:
    info = json.load(f)

print("格式版本:", info["codebase_version"])
print("Episodes:", info["total_episodes"])
print("Frames:", info["total_frames"])
print("Tasks:", info["total_tasks"])
print("FPS:", info["fps"])

print("\nParquet 文件数:", len(list(dataset_dir.glob("data/**/*.parquet"))))
print("MP4 文件数:", len(list(dataset_dir.glob("videos/**/*.mp4"))))

print("\n完整数据集下载完成")

Dataset: lerobot/libero
Revision: a1aaacb7f6cd6ee5fb43120f673cebb0cfea7dd4

下载目录: /content/datasets/libero
格式版本: v3.0
Episodes: 1693
Frames: 273465
Tasks: 40
FPS: 10.0

Parquet 文件数: 377
MP4 文件数: 74

完整数据集下载完成


Fetching 457 files: 100%|██████████| 457/457 [00:35<00:00, 12.82it/s]


## 4.登录HuggingFace

In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


## 5.登录W&B

In [17]:
import wandb
import os

os.environ["WANDB_API_KEY"] = "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"

result = wandb.login()

print("W&B 登录成功:", result)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: lemons916332 (lemons916332-zhejiang-gongshang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B 登录成功: True


## 6.训练

In [19]:
import os
import shlex
import shutil
import subprocess
from pathlib import Path
from google.colab import runtime

# ============================================================
# 1. 训练配置
# ============================================================

cmd = shlex.split("""
    /content/.venv/bin/python -m lerobot.scripts.lerobot_train

    --dataset.repo_id=lerobot/libero
    --dataset.root=/content/datasets/libero
    --dataset.video_backend=torchcodec
    --dataset.eval_split=0.0

    --policy.path=lerobot/smolvla_base
    --policy.input_features=null
    --policy.device=cuda
    --policy.repo_id=lemons2027/smolvla-libero-baseline-20k
    --policy.push_to_hub=true
    --policy.private=true

    --batch_size=64
    --steps=20000
    --num_workers=2
    --log_freq=50
    --eval_steps=0
    --env_eval_freq=0
    --save_checkpoint=true
    --save_freq=2000
    --seed=42

    --job_name=smolvla_libero_baseline_20k
    --output_dir=/content/outputs/smolvla_libero_baseline_20k
    --save_checkpoint_to_hub=true

    --wandb.enable=true
    --wandb.project=smolvla-3d-libero
    --wandb.mode=online
    --wandb.disable_artifact=true
""")

# ============================================================
# 2. 日志与环境配置
# ============================================================

log_path = Path("/content/smolvla_training.log")
log_path.parent.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()

# 尝试关闭 tqdm 的逐 Step 进度条，减少日志文件大小
env["TQDM_DISABLE"] = "1"

# ============================================================
# 3. 开始训练
# ============================================================

print("=" * 60)
print("SmolVLA + LIBERO Baseline")
print("训练步数：20000")
print("Batch Size：64")
print("日志文件：", log_path)
print("训练指标：W&B / smolvla-3d-libero")
print("=" * 60)
print("训练已启动，Notebook 不会持续打印进度条。", flush=True)

# 所有训练输出写入文件，不输出到 Notebook
with log_path.open("w", encoding="utf-8") as log_file:
    result = subprocess.run(
        cmd,
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        check=False,
    )

# ============================================================
# 4. 训练结束，处理结果
# ============================================================

print("\n训练进程已退出")
print("退出码：", result.returncode, flush=True)

# 如果 Google Drive 已挂载，则备份日志
drive_dir = Path("/content/drive/MyDrive/SmolVLA/logs")

if Path("/content/drive/MyDrive").is_dir():
    try:
        drive_dir.mkdir(parents=True, exist_ok=True)
        backup_path = drive_dir / log_path.name
        shutil.copy2(log_path, backup_path)
        print("日志已备份至：", backup_path)
    except Exception as e:
        print("Google Drive 日志备份失败：", e)

# ============================================================
# 5. 成功自动释放 GPU，失败保留环境
# ============================================================

if result.returncode == 0:

    print("\n训练成功完成！", flush=True)
    print("模型应已按训练配置执行 Hugging Face 上传。")
    print("请在 Hub 确认模型和检查点是否完整。")
    print("准备自动释放 Colab GPU...", flush=True)

    runtime.unassign()

else:

    print("\n训练失败，保留 Colab 运行时供排查。")

    print("\n========== 最后 50 行训练日志 ==========")

    lines = log_path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    print("\n".join(lines[-50:]))

    print("\n完整日志位置：", log_path)


SmolVLA + LIBERO Baseline
训练步数：20000
Batch Size：64
日志文件： /content/smolvla_training.log
训练指标：W&B / smolvla-3d-libero
训练已启动，Notebook 不会持续打印进度条。

训练进程已退出
退出码： 0

训练成功完成！
模型应已按训练配置执行 Hugging Face 上传。
请在 Hub 确认模型和检查点是否完整。
准备自动释放 Colab GPU...
